In [5]:
import time
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter, Retry
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path

BASE_URL = "https://wehi-rcpstudentinternship.github.io/"


def make_session():
    s = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.8,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.headers.update(
        {
            "User-Agent": "Mozilla/5.0 (compatible; student-scraper/1.0)",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }
    )
    return s


def fetch_html(session, url, timeout=20):
    """
    Fetch with simple fallbacks:
    - try url as-is
    - if 404 and no .html, try adding .html
    - if 404 and no trailing slash, try adding /
    """
    candidates = [url]
    if not url.endswith(".html") and not url.endswith("/"):
        candidates.append(url + ".html")
    if not url.endswith("/"):
        candidates.append(url + "/")

    last_resp, last_tried = None, None
    for u in candidates:
        resp = session.get(u, timeout=timeout, allow_redirects=True)
        last_resp, last_tried = resp, u
        if resp.status_code == 200:
            return resp, resp.url  # final_url (after redirects)

    raise requests.HTTPError(f"HTTP {last_resp.status_code} for {last_tried}")


def scrape_web_page(session, url, topic="General"):
    resp, final_url = fetch_html(session, url, timeout=20)
    soup = BeautifulSoup(resp.text, "html.parser")

    documents = []

    # Primary: h2 sections. Fallback: h1 sections (for simple pages)
    titles = soup.find_all("h2")
    if not titles:
        titles = soup.find_all("h1")

    for title in titles:
        heading = title.get_text(" ", strip=True)
        if not heading:
            continue

        content_parts = [heading]
        sib = title.find_next_sibling()

        # Stop when next heading starts (h1 or h2)
        while sib and sib.name not in ["h1", "h2"]:
            if sib.name in ["p", "ul", "ol", "div", "pre", "blockquote", "table"]:
                text = sib.get_text(" ", strip=True)
                if text:
                    content_parts.append(text)
            sib = sib.find_next_sibling()

        section_id = title.get("id")
        source = f"{final_url}#{section_id}" if section_id else final_url

        documents.append(
            {
                "title": heading,
                "content": " ".join(content_parts).strip(),
                "topic": topic,
                "source": source,
            }
        )

    return documents


def add_keywords_tfidf(docs, top_n=5):
    if not docs:
        return docs

    corpus = [d["content"] for d in docs]
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(corpus)
    terms = vectorizer.get_feature_names_out()

    for i, d in enumerate(docs):
        row = X[i].toarray().ravel()
        top_idx = row.argsort()[-top_n:][::-1]
        d["keywords"] = [terms[j] for j in top_idx if row[j] > 0]

    return docs


if __name__ == "__main__":
    links = [
        {"path": "", "topic": "Unpaid Student Internship Program"},  # homepage (students)
        {"path": "complex-projects.html", "topic": "complex ambiguous projects"},
        {"path": "software_maturity_model.html", "topic": "software maturity model"},
        {"path": "explanation_about_ohs.html", "topic": "explanation about ohs"},
        {"path": "top-5-mistakes.html", "topic": "top 5 mistakes"},
        {"path": "project-wikis.html", "topic": "project wikis"},
        {"path": "student-loxcoder.html", "topic": "loxcoder"},
        {"path": "student-data-commons.html", "topic": "student data commons"},
        {"path": "student-cryoem.html", "topic": "cryoem"},
        {"path": "student-genomics-qc.html", "topic": "genomics qc"},
        {"path": "student-schex.html", "topic": "schex"},
        {"path": "student-mixOmics.html", "topic": "mixOmics"},
        {"path": "student-capacity-planning.html", "topic": "capacity planning"},
        {"path": "student-haemosphere.html", "topic": "haemosphere"},
        {"path": "student-imaging.html", "topic": "imaging"},
        {"path": "student-quantum.html", "topic": "quantum"},
        {"path": "student-genomics-metadata.html", "topic": "genomics metadata"},
        {"path": "student-aive.html", "topic": "aive"},
        {"path": "student-bionix.html", "topic": "bionix"},
        {"path": "student-clinical-dashboards.html", "topic": "clinical dashboards"},
        {"path": "email_acknowledgement.html", "topic": "email acknowledgement"},
        {"path": "code-of-conduct.html", "topic": "code of conduct"},
        {"path": "faq.html", "topic": "FAQ"},
    ]

    session = make_session()

    all_docs = []
    failed = []

    for link in links:
        url = urljoin(BASE_URL, link["path"])
        try:
            docs = scrape_web_page(session, url, topic=link["topic"])
            all_docs.extend(docs)
            print(f"OK   {url}  ->  {len(docs)} sections")
        except Exception as e:
            failed.append(url)
            print(f"FAIL {url}  ->  {e}")
        time.sleep(0.3)

    all_docs = add_keywords_tfidf(all_docs, top_n=5)

    # Notebook-safe output path (works in Jupyter + normal python)
    out_dir = Path.cwd() / "data"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "aggregated_data.json"

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(all_docs, f, ensure_ascii=False, indent=2)

    print(f"\nSaved {len(all_docs)} sections to: {out_file}")

    if failed:
        print("\nThese URLs failed:")
        for u in failed:
            print(" -", u)

OK   https://wehi-rcpstudentinternship.github.io/  ->  4 sections
OK   https://wehi-rcpstudentinternship.github.io/complex-projects.html  ->  2 sections
OK   https://wehi-rcpstudentinternship.github.io/software_maturity_model.html  ->  2 sections
OK   https://wehi-rcpstudentinternship.github.io/explanation_about_ohs.html  ->  7 sections
OK   https://wehi-rcpstudentinternship.github.io/top-5-mistakes.html  ->  8 sections
OK   https://wehi-rcpstudentinternship.github.io/project-wikis.html  ->  4 sections
OK   https://wehi-rcpstudentinternship.github.io/student-loxcoder.html  ->  4 sections
OK   https://wehi-rcpstudentinternship.github.io/student-data-commons.html  ->  5 sections
OK   https://wehi-rcpstudentinternship.github.io/student-cryoem.html  ->  4 sections
OK   https://wehi-rcpstudentinternship.github.io/student-genomics-qc.html  ->  4 sections
OK   https://wehi-rcpstudentinternship.github.io/student-schex.html  ->  5 sections
OK   https://wehi-rcpstudentinternship.github.io/studen

In [6]:
# check working directory
import os
print("Current working directory:", os.getcwd())


Current working directory: /Users/hanshitang/sem1-25-student-organiser-rag-llm
